In [10]:
import pandas as pd
from openpyxl import Workbook
import openpyxl
from openpyxl.styles import NamedStyle, numbers
import os
from datetime import datetime
import glob
import dateutil.parser
import locale

print(locale.getlocale())  # Check system locale

# Define the types and their corresponding descriptions
types = {
    "SBE": "Small Business",
    "SDB": "Small Disadvantaged Business",
    "MBE": "Minority Owned Business",
    "CAB": "Certified Aboriginal Business",
    "WBE": "Woman Owned Business",
    "WOSB": "Woman Owned Small Business",
    "VBE": "Veteran Owned Business",
    "SDVBE": "Service Disabled Veteran Owned Business",
    "VOSB": "Veteran Owned Small Business",
    "SDVOSB": "Service Disabled Veteran Owned Small Business",
    "LGBTBE": "LGBTQ+ Owned Business",
    "PWD": "Persons With Disabilities Owned Business",
    "DBE": "Disadvantaged Business Enterprise",
    "HUB": "Hub Zone Business",
    "BCORP": "Benefit Corporation",
    "HBCU": "Historically Black College or University",
    "ANC": "Alaskan Native Corporation",
    "ANCSBE": "ANC Small Business"
}

def process_type(type_key, teal_iq_df, vm_analysis_df, rules_df, exceptions_df):
    print(f"Processing type: {type_key}")
    print("Rules tab preview:")
    print(rules_df.iloc[:10, :3])  # Display the first 10 rows and first 3 columns

    # Step 1: Filter for the specific type
    temp_df = teal_iq_df[teal_iq_df['subcategory'].str.lower() == type_key.lower()].copy()
    print(f"Step 1 - After filtering by subcategory '{type_key}': {len(temp_df)} rows")

    # Step 2: Remove rows with excluded certifying bodies
    certifying_bodies = rules_df.iloc[14:100, 0].dropna().str.lower().tolist()
    temp_df = temp_df[~temp_df['certifying_body'].str.lower().isin(certifying_bodies)]
    print(f"Step 2 - After filtering by certifying bodies: {len(temp_df)} rows")

    # Step 3: Remove rows where tealbook_id matches in the Exceptions tab
    exceptions_ids = exceptions_df['tealbook_id'].dropna().unique()
    temp_df = temp_df[~temp_df['tealbook_id'].isin(exceptions_ids)]
    print(f"Step 3 - After removing rows matching Exceptions tealbook_id: {len(temp_df)} rows")
    
    # Step 4: Filter by expiration_date_ref
    expiration_date_ref = pd.to_datetime(rules_df.iloc[2, 1], format='%Y-%m-%d', errors='coerce')
    print(f"Raw expiration_date_ref value: {rules_df.iloc[2, 1]}")
    print(f"Parsed expiration_date_ref: {expiration_date_ref}")
    
    # Read potential_date_ref from row 2, column 2
    potential_date_ref = pd.to_datetime(rules_df.iloc[3, 1], format='%Y-%m-%d', errors='coerce')
    print(f"Raw potential_date_ref value: {rules_df.iloc[3, 1]}")
    print(f"Parsed potential_date_ref: {potential_date_ref}")

    # Step 4: Process expiration_date filtering and Use column assignment
    
    # Convert expiration_date column to datetime
    temp_df['expiration_date'] = pd.to_datetime(temp_df['expiration_date'], format='%m/%d/%Y', errors='coerce')
    
    # Print debugging info
    print(f"Step 4 - Valid expiration_date values: {temp_df['expiration_date'].notna().sum()} rows")
    
    # Remove rows where expiration_date is **before** potential_date_ref
    temp_df = temp_df[(temp_df['expiration_date'].isna()) | (temp_df['expiration_date'] >= potential_date_ref)]
    print(f"Step 4.1 - After removing rows before potential_date_ref ({potential_date_ref}): {len(temp_df)} rows")
     
    # Handle NaT cases
    if pd.isna(expiration_date_ref) or pd.isna(potential_date_ref):
        print("⚠️ Error: One or both date references could not be parsed. Please check the Rules tab.")
    
    # Step 5: Convert expiration_date column
    temp_df['expiration_date'] = pd.to_datetime(temp_df['expiration_date'], format='%m/%d/%Y', errors='coerce')
    print(f"🔹 Step 5 - Valid expiration_date values: {temp_df['expiration_date'].notna().sum()} rows")

    # Step 6: Remove rows where expiration_date is **before** potential_date_ref
    temp_df = temp_df[(temp_df['expiration_date'].isna()) | (temp_df['expiration_date'] >= potential_date_ref)]
    print(f"🔹 Step 6 - After removing rows before potential_date_ref ({potential_date_ref}): {len(temp_df)} rows")

    # Step 7: Assign "Use" column based on expiration_date value
    def assign_use(row):
        if pd.isna(row['expiration_date']) or row['expiration_date'] >= expiration_date_ref:
            return "Yes"  # ✅ Blank expiration_date or >= expiration_date_ref → "Yes"
        elif expiration_date_ref > row['expiration_date'] >= potential_date_ref:
            return "Potential"  # ✅ Between expiration_date_ref & potential_date_ref → "Potential"
        return None  # This should never be reached

    temp_df['Use'] = temp_df.apply(assign_use, axis=1)

    # Debugging printout for Use column
    print(f"🔹 Step 7 - Rows marked as 'Yes': {sum(temp_df['Use'] == 'Yes')}")
    print(f"🔹 Step 7 - Rows marked as 'Potential': {sum(temp_df['Use'] == 'Potential')}")

    # If temp_df is empty, skip further processing
    if len(temp_df) == 0:
        print(f"⚠️ No rows left after filtering for type: {type_key}")
        return 0, 0, 0, 0, temp_df  # Include temp_df (empty DataFrame) in return

    # Step 8: Deduplicate on internal_supplier_id, preferring blank expiration_date, then latest date
    temp_df['expiration_date_is_blank'] = temp_df['expiration_date'].isna()  # Boolean flag for NaT values
    
    # Sort by:
    # 1. expiration_date_is_blank (True first, meaning blank dates are prioritized)
    # 2. expiration_date descending (keeping the most recent date for non-blanks)
    temp_df = temp_df.sort_values(by=['internal_supplier_id', 'expiration_date_is_blank', 'expiration_date'], 
                                  ascending=[True, False, False])
    
    # Deduplicate on internal_supplier_id, keeping the first occurrence (which now prefers blanks)
    temp_df = temp_df.drop_duplicates(subset=['internal_supplier_id'], keep='first')
    
    # Drop the helper column
    temp_df = temp_df.drop(columns=['expiration_date_is_blank'])
    
    print(f"🔹 Step 8 - After deduplication (keeping blank expiration_date where available): {len(temp_df)} rows")

    # Step 9: Merge with VM Analysis Spend Summary
    temp_df = temp_df.merge(vm_analysis_df, how='left', left_on='internal_supplier_id', right_on='internal supplier id')
    print(f"🔹 Step 9 - After merging with spend summary: {len(temp_df)} rows")

    # Step 10: Ensure aggregated spend is valid (not zero, blank, or negative)
    valid_spend_df = temp_df[temp_df['aggregated spend'].notna() & (temp_df['aggregated spend'] != 0)]
    
    # Step 11: Calculate Qualified Count and Spend
    qualified_count = valid_spend_df[valid_spend_df['Use'] == 'Yes'].shape[0]
    qualified_spend = valid_spend_df[valid_spend_df['Use'] == 'Yes']['aggregated spend'].sum()
    
    # Step 12: Calculate Potential Count and Spend
    potential_count = valid_spend_df[valid_spend_df['Use'] == 'Potential'].shape[0]
    potential_spend = valid_spend_df[valid_spend_df['Use'] == 'Potential']['aggregated spend'].sum()

    # Print final metrics
    print(f"🔹 Final Metrics for '{type_key}':")
    print(f"   ✅ Qualified Count: {qualified_count}, Qualified Spend: {qualified_spend}")
    print(f"   ⚡ Potential Count: {potential_count}, Potential Spend: {potential_spend}")
    
    return qualified_count, qualified_spend, potential_count, potential_spend, temp_df

    def parse_date_with_fallback(date_str):
        try:
            # Attempt to parse in MM-DD-YYYY format
            return pd.to_datetime(date_str, format='%m-%d-%Y', errors='coerce')
        except:
            # Fallback to automatic parsing
            return pd.to_datetime(date_str, errors='coerce')
    
    expiration_date_ref = parse_date_with_fallback(rules_df.iloc[2, 1])
    potential_date_ref = parse_date_with_fallback(rules_df.iloc[3, 1])

    print(f"Raw expiration_date_ref value: {rules_df.iloc[2, 1]}")
    print(f"Raw potential_date_ref value: {rules_df.iloc[3, 1]}")
    print(f"Raw value in B4 (expiration_date_ref): {rules_df.iloc[3, 1]}")


# Function to find files by pattern
def find_file_by_pattern(pattern):
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(f"No file found matching pattern: {pattern}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files found matching pattern: {pattern}")
    return matches[0]

# Calculate Diverse, Total, and Percentage values
def calculate_summary(teal_iq_file):
    # Load Diversity tab
    diversity_df = pd.read_excel(teal_iq_file, sheet_name='Diversity', engine='openpyxl')

    # Load Spend tab
    spend_df = pd.read_excel(teal_iq_file, sheet_name='Spend', engine='openpyxl')

    # Qualified Count calculations
    diverse_count = diversity_df[diversity_df['Qualified'].str.lower() == 'yes'].shape[0]
    total_count = diversity_df[diversity_df['Spend'] > 0].shape[0]
    percentage_count = round((diverse_count / total_count) * 100, 2) if total_count > 0 else 0

    # Qualified Spend calculations
    diverse_spend = diversity_df.loc[diversity_df['Qualified'].str.lower() == 'yes', 'Spend'].sum()
    total_spend = spend_df['Spend Amount*'].sum()
    percentage_spend = round((diverse_spend / total_spend) * 100, 2) if total_spend > 0 else 0

    return {
        "Qualified Count": [diverse_count, total_count, percentage_count],
        "Qualified Spend": [diverse_spend, total_spend, percentage_spend],
        "Labels": ["Diverse", "Total", "Percentage"]
    }

# Populate the Qualified Percentage column
def populate_qualified_percentage(output_data):
    # Locate the Total row's Qualified Spend value
    total_row = next((row for row in output_data if row[0] == "Total"), None)
    if not total_row or not total_row[3]:
        print("Error: Total row not found or Qualified Spend is zero. Cannot calculate percentages.")
        return

    total_qualified_spend = total_row[3]
    print(f"Total Qualified Spend: {total_qualified_spend}")

    # Calculate percentages for rows from SBE to ANCSBE
    for row in output_data:
        print(f"Processing row: {row[0]} with Qualified Spend: {row[3]}")  # Debugging
        if row[0] in types.keys():  # Only process rows in the types dictionary
            if row[3] is not None and total_qualified_spend > 0:
                row[4] = round((row[3] / total_qualified_spend) * 100, 2)  # Calculate percentage
                print(f"Updated Qualified Percentage for {row[0]}: {row[4]}%")
            else:
                row[4] = None  # Leave blank if Qualified Spend is None or zero

# Main function
def main():
    # Generate today's date in the desired format (e.g., YYYY-MM-DD)
    today_date = datetime.now().strftime('%Y-%m-%d')

    # Create the output file name
    output_file = f"SDP - Diversity Summary tab - {today_date}.xlsx"
    
    # Step 1: Load input files
    teal_iq_file = find_file_by_pattern("*Teal iQ*.xlsm")
    vm_analysis_file = find_file_by_pattern("*VM Analysis*.xlsx")
    teal_iq_df = pd.read_excel(teal_iq_file, sheet_name='Certificates', engine='openpyxl')
    vm_analysis_df = pd.read_excel(vm_analysis_file, sheet_name='Spend Summary')
    rules_df = pd.read_excel(teal_iq_file, sheet_name='Rules', engine='openpyxl')
    exceptions_df = pd.read_excel(teal_iq_file, sheet_name='Exceptions', engine='openpyxl')  # Load Exceptions tab

    # Step 2: Read exclusion types from B3 in the Rules tab
    exclusion_types_raw = rules_df.iloc[1, 1]  # Read cell B3
    if pd.notna(exclusion_types_raw) and isinstance(exclusion_types_raw, str):
        exclusion_types = [x.strip().lower() for x in exclusion_types_raw.split(',')]
    else:
        exclusion_types = []
    
    print(f"Exclusion Types from B3: {exclusion_types}")  # Debugging

    # Step 3: Process each type and generate the main output data
    output_data = []
    sbe_data = None
    mbe_data = None
    vbe_data = None

    for type_key, description in types.items():
        # Pass the exceptions_df as an argument
        qualified_count, qualified_spend, potential_count, potential_spend, processed_df = process_type(
            type_key, teal_iq_df, vm_analysis_df, rules_df, exceptions_df
        )
        output_data.append([type_key, description, qualified_count, qualified_spend, None, potential_count, potential_spend])

        # Store SBE, MBE, and VBE data separately
        if type_key == "SBE":
            sbe_data = processed_df.copy()
        elif type_key == "MBE":
            mbe_data = processed_df.copy()
        elif type_key == "VBE":
            vbe_data = processed_df.copy()

    # Step 4: Apply Exclusion Types
    for row in output_data:
        type_key = row[0].lower()
        if type_key in exclusion_types:
            print(f"Excluding Type: {row[0]}")
            row[2] = 0  # Qualified Count
            row[3] = 0  # Qualified Spend
            row[4] = None  # Qualified Percentage
            row[5] = 0  # Potential Count
            row[6] = 0  # Potential Spend

    # Step 5: Calculate and append Diverse, Total, and Percentage rows
    summary_metrics = calculate_summary(teal_iq_file)
    output_data.append(["", "", "", "", "", "", ""])  # Blank row
    for label, count, spend in zip(
        summary_metrics["Labels"],
        summary_metrics["Qualified Count"],
        summary_metrics["Qualified Spend"]
    ):
        output_data.append([label, "", count, spend, None, 0, 0])
    
    # Step 6: Populate Qualified Percentage column
    populate_qualified_percentage(output_data)

    # Step 7: Create the output DataFrame
    output_df = pd.DataFrame(output_data, columns=[
        'Type', 'Description', 'Qualified Count', 'Qualified Spend', 'Qualified Percentage', 'Potential Count', 'Potential Spend'
    ])

    # Step 8: Save to Excel
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        output_df.to_excel(writer, index=False, sheet_name='Output')

        # Add individual tabs for SBE, MBE, and VBE
        if sbe_data is not None and not sbe_data.empty:
            sbe_data.to_excel(writer, index=False, sheet_name='SBE')
        if mbe_data is not None and not mbe_data.empty:
            mbe_data.to_excel(writer, index=False, sheet_name='MBE')
        if vbe_data is not None and not vbe_data.empty:
            vbe_data.to_excel(writer, index=False, sheet_name='VBE')

    # Step 9: Apply formatting
    apply_percentage_formatting(output_file)

    print(f"Output saved to {output_file}")

# Apply percentage formatting
def apply_percentage_formatting(output_file):
    wb = openpyxl.load_workbook(output_file)
    ws = wb['Output']

    # Format the Qualified Percentage column
    qualified_percentage_col = 5  # Column index for "Qualified Percentage" (1-based)
    for row in range(2, ws.max_row + 1):  # Skip header row
        cell = ws.cell(row=row, column=qualified_percentage_col)
        if isinstance(cell.value, (int, float)):
            cell.value = cell.value / 100  # Convert to decimal for percentage
            cell.number_format = "0.00%"

    # Format the Percentage row for Qualified Count and Qualified Spend
    for row in ws.iter_rows(min_row=ws.max_row, max_row=ws.max_row, min_col=3, max_col=4):
        for cell in row:
            if isinstance(cell.value, (int, float)):
                cell.value = cell.value / 100  # Convert to decimal for percentage
                cell.number_format = "0.00%"

    # Adjust column widths
    column_widths = {
        'A': 15,  # Column A
        'B': 25,  # Column B
        'C': 15,  # Column C
        'D': 20,  # Column D
        'E': 15,  # Column E
        'F': 15,  # Column F
        'G': 15,  # Column G
    }
    for col_letter, width in column_widths.items():
        ws.column_dimensions[col_letter].width = width

    # Save the workbook
    wb.save(output_file)
    print(f"Percentage formatting and column widths applied to {output_file}")


# Example usage
if __name__ == "__main__":
    main()

('en_US', 'UTF-8')
Exclusion Types from B3: ['bcorp', 'hub', 'lsa', 'hud', 'sbe', 'sdb', 'vosb']
Processing type: SBE
Rules tab preview:
                                     Reporting Rules  \
0                                                NaN   
1  Exclude Subcategories (provide comma delimited...   
2                     Expired prior to are potential   
3                   Expired prior to are not diverse   
4                                                NaN   
5                                                NaN   
6                                                NaN   
7                                                NaN   
8                                                NaN   
9                                                NaN   

                       Unnamed: 1      Unnamed: 2  
0                             NaN             NaN  
1  bcorp,hub,lsa,hud,sbe,sdb,vosb         eg: sbe  
2             2023-10-01 00:00:00  eg: 2024-01-01  
3             2022-10-01 00:00:00     